# Black–Scholes value, delta, and vega from a volatility surface (SPX)

This notebook reads `USD-SURF-SPX.csv` (a volatility surface by strike and expiry) and computes, **for each row/grid-point**:

- Option **value** (European call + European put)
- **Delta** (call delta + put delta)
- **Vega** (sensitivity to volatility)

Outputs are saved as:
- `USD-SURF-SPX_BS_value_delta_vega.xlsx`

---

## What we assume (inputs per row)

We use these columns (names must match the CSV):

- Spot S            = `Spot Price`
- Strike K          = `Strike Price`
- Vol sigma         = `Volatility`  (annualized, in decimals; e.g. 0.20 = 20%)
- Valuation date    = `ValuationDate`
- Expiration date   = `Expiration Date`
- Discount factor DF= `Discount Factor`  (continuous discounting)
- Dividend yield q  = `Dividend Yield`   (continuous)

Time to expiry:

- T = (Expiration Date − ValuationDate) in **years** using ACT/365.

Risk-free rate implied by DF:

- DF = exp(−r*T)  =>  r = −ln(DF)/T

---

## Black–Scholes formulas (plain text)

Let:

- d1 = [ ln(S/K) + (r − q + 0.5*sigma^2)*T ] / [ sigma*sqrt(T) ]
- d2 = d1 − sigma*sqrt(T)

Where:
- N(x) is the standard normal CDF
- n(x) is the standard normal PDF

Call value:
- C = S*exp(−q*T)*N(d1) − K*exp(−r*T)*N(d2)

Put value:
- P = K*exp(−r*T)*N(−d2) − S*exp(−q*T)*N(−d1)

Call delta:
- Delta_call = exp(−q*T)*N(d1)

Put delta:
- Delta_put  = exp(−q*T)*(N(d1) − 1)

Vega (same for call/put in BS):
- Vega = S*exp(−q*T)*n(d1)*sqrt(T)

Notes:
- The Vega above is **per 1.00 change in volatility** (e.g., from 0.20 to 0.21 is +0.01).
- Vega per 1% vol move = 0.01 * Vega


In [85]:
import pandas as pd
import numpy as np
from scipy.stats import norm

INPUT_PATH = "USD-SURF-SPX.csv"   # if running locally, put the csv next to this notebook
df = pd.read_csv(INPUT_PATH)

df.head()


,ValuationDate,Underlying,ReferenceType,Spot Price,Expiration Date,Relative Strike,Strike Price,Volatility,Discount Factor,Forward Price,Dividend Yield
0,30-09-2022,S&P 500,Future,3585.62,21-10-2022,50,1792.810,0.823446,0.998239,3588.952838,0.014497
1,30-09-2022,S&P 500,Future,3585.62,21-10-2022,60,2151.372,0.726379,0.998239,3588.952838,0.014497
2,30-09-2022,S&P 500,Future,3585.62,21-10-2022,70,2509.934,0.634870,0.998239,3588.952838,0.014497
3,30-09-2022,S&P 500,Future,3585.62,21-10-2022,80,2868.496,0.487238,0.998239,3588.952838,0.014497
4,30-09-2022,S&P 500,Future,3585.62,21-10-2022,90,3227.058,0.360328,0.998239,3588.952838,0.014497


## Step 1) Parse dates and compute T, r, q

We convert the date columns to datetime and compute:

- T (years) = (expiry - valuation) / 365
- r from DF: r = -ln(DF)/T
- q is taken directly from the CSV


In [87]:
# Parse dates
df["ValuationDate"] = pd.to_datetime(df["ValuationDate"])
df["Expiration Date"] = pd.to_datetime(df["Expiration Date"])

# Time to expiry (years, ACT/365)
df["T"] = (df["Expiration Date"] - df["ValuationDate"]).dt.days / 365.0

# Guard against non-positive T (should not happen in a clean surface)
if (df["T"] <= 0).any():
    bad = df.loc[df["T"] <= 0, ["ValuationDate", "Expiration Date", "T"]].head(10)
    raise ValueError("Found non-positive T in these rows:\n" + bad.to_string(index=False))

# Implied r from discount factor DF = exp(-rT)
df["r"] = -np.log(df["Discount Factor"]) / df["T"]

# Dividend yield q (continuous)
df["q"] = df["Dividend Yield"]

df[["T", "r", "q"]].describe()


C:\Users\Deepak\AppData\Local\Temp\ipykernel_11844\2540853061.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["ValuationDate"] = pd.to_datetime(df["ValuationDate"])
C:\Users\Deepak\AppData\Local\Temp\ipykernel_11844\2540853061.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Expiration Date"] = pd.to_datetime(df["Expiration Date"])


,T,r,q
count,299.000000,299.000000,299.000000
mean,5.840143,0.037128,0.010949
std,5.367090,0.003348,0.004075
min,0.057534,0.030635,0.006108
25%,0.958904,0.034745,0.007328
50%,4.219178,0.036247,0.009018
75%,10.221918,0.040427,0.014497
max,19.235616,0.042409,0.020384


## Step 2) Compute d1, d2, value, delta, vega for each row

In [89]:
S = df["Spot Price"].to_numpy(dtype=float)
K = df["Strike Price"].to_numpy(dtype=float)
T = df["T"].to_numpy(dtype=float)
r = df["r"].to_numpy(dtype=float)
q = df["q"].to_numpy(dtype=float)
sigma = df["Volatility"].to_numpy(dtype=float)

sqrtT = np.sqrt(T)
d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
d2 = d1 - sigma * sqrtT

disc_r = np.exp(-r * T)
disc_q = np.exp(-q * T)

# Values
df["Call Value"] = S * disc_q * norm.cdf(d1) - K * disc_r * norm.cdf(d2)
df["Put Value"]  = K * disc_r * norm.cdf(-d2) - S * disc_q * norm.cdf(-d1)

# Deltas
df["Call Delta"] = disc_q * norm.cdf(d1)
df["Put Delta"]  = disc_q * (norm.cdf(d1) - 1.0)

# Vega (per 1.00 vol)
df["Vega"] = S * disc_q * norm.pdf(d1) * sqrtT
df["Vega (per 1% vol)"] = 0.01 * df["Vega"]

df[["Call Value","Put Value","Call Delta","Put Delta","Vega","Vega (per 1% vol)"]].head()


,Call Value,Put Value,Call Delta,Put Delta,Vega,Vega (per 1% vol)
0,1793.005248,0.027535,0.999015,-0.000151,0.502090,0.005021
1,1435.276541,0.229400,0.997921,-0.001245,3.539399,0.035394
2,1078.561912,1.445344,0.991507,-0.007660,18.142711,0.181427
3,723.136661,3.950665,0.975092,-0.024074,48.692310,0.486923
4,376.737152,15.481729,0.897741,-0.101426,152.470354,1.524704


## Step 3) Save the results

We save:
- CSV (easy to submit / re-open anywhere)
- Excel (nice for viewing)



In [91]:
OUT_XLSX = "USD-SURF-SPX_BS_value_delta_vega.xlsx"
df.to_excel(OUT_XLSX, index=False)
print("Saved:", OUT_XLSX)

Saved: USD-SURF-SPX_BS_value_delta_vega.xlsx


## Quick sanity checks (optional)

A few quick checks you can do:

1) Put–Call parity (should be close):
   C - P ≈ S*exp(-qT) - K*exp(-rT)

2) Vega should be positive for T>0 and sigma>0.



In [93]:
# Put–Call parity check
lhs = df["Call Value"] - df["Put Value"]
rhs = df["Spot Price"] * np.exp(-df["q"]*df["T"]) - df["Strike Price"] * np.exp(-df["r"]*df["T"])
parity_error = (lhs - rhs).abs()

print("Put-Call parity |error| summary:")
print(parity_error.describe())

# Vega positivity check
print("\nAny negative Vega?", (df["Vega"] < 0).any())


Put-Call parity |error| summary:
count    2.990000e+02
mean     1.861194e-13
std      1.660691e-13
min      0.000000e+00
25%      0.000000e+00
50%      2.273737e-13
75%      2.273737e-13
max      6.821210e-13
dtype: float64

Any negative Vega? False
